# Native PyTorch tutorial 4

Full outputs, selected Jacobians/Hessians, analytic numerical checks, zero higher derivatives and differentiable derivative penalties.

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial4"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# How to get the gradients: dy/dx?

- Note: 
    - since we wrap up everything in torch.nn.Module, we want to keep things as simple as possible as well for getting the gradient

## Construct a torch.nn.Module 

In [ ]:
model = dense_tanh(4,5).to(device)


## Import `JacobianLayer` and `HessianLayer`

In [ ]:
from nif.layers import JacobianLayer, HessianLayer


## You can also decide what indices of both output $y$ and input $x$ that you want to get the gradients on

- here you define them as `y_index` and `x_index`

In [ ]:
x_index, y_index = [0,1,2,3], [0,1,2,3,4]


## Now we can use `nif.layers.JacobianLayer` to **wrap** up the `torch.nn.Module` 

In [ ]:
y_and_dydx_layer = JacobianLayer(model,y_index,x_index)


# Get the gradient tensor by passing input to the `JacobianLayer` instance
- you can see there are two outputs, first one is `y`, second one is `dy/dx`

In [ ]:
x_data=torch.rand(10,4,device=device)
y_pred,dydx=y_and_dydx_layer(x_data)


## We can also verify the shape, it is `(batch_size, dim(y_chosen), dim(x_chosen))`

In [ ]:
assert y_pred.shape == (10,5) and dydx.shape == (10,5,4)
print("f(x) shape",y_pred.shape,"df(x)/dx shape",dydx.shape)


# We can even compute Hessian, the second order partial derivatives, for any `torch.nn.Module`

In [ ]:
y_and_dydx_and_dy2dx2_layer=HessianLayer(model,y_index,x_index)
y_pred,dydx,dy2dx2=y_and_dydx_and_dy2dx2_layer(x_data)


## we can also verify the shape, it is `(batch_size, dim(y_chosen), dim(x_chosen), dim(x_chosen))`

In [ ]:
assert dy2dx2.shape == (10,5,4,4)
print("d2f(x)/dx2 shape",dy2dx2.shape)
# Analytic tanh(linear(x)) Jacobian and Hessian validate actual values.
w=model[0].weight
expected_j=(1-y_pred.square())[:,:,None]*w[None,:,:]
expected_h=(-2*y_pred*(1-y_pred.square()))[:,:,None,None]*w[None,:,:,None]*w[None,:,None,:]
torch.testing.assert_close(dydx,expected_j)
torch.testing.assert_close(dy2dx2,expected_h)
# Selecting derivatives does not truncate the full output.
y_full,j_selected,h_selected=HessianLayer(model,[4,1],[3,0])(x_data)
assert y_full.shape == (10,5)
torch.testing.assert_close(j_selected,dydx[:,[4,1]][:,:,[3,0]])
torch.testing.assert_close(h_selected,dy2dx2[:,[4,1]][:,:,[3,0]][:,:,:,[3,0]])
# Linear/constant functions have valid zero higher derivatives.
linear=nn.Linear(4,5).to(device)
_,_,h_linear=HessianLayer(linear,[0],[0,1])(x_data)
torch.testing.assert_close(h_linear,torch.zeros_like(h_linear))
class Constant(nn.Module):
    def __init__(self):
        super().__init__(); self.value=nn.Parameter(torch.ones(5))
    def forward(self,x): return self.value.expand(len(x),-1)
constant=Constant().to(device)
_,j_constant,h_constant=HessianLayer(constant,[0],[0,1])(x_data)
torch.testing.assert_close(j_constant,torch.zeros_like(j_constant))
torch.testing.assert_close(h_constant,torch.zeros_like(h_constant))
# Derivative penalties must differentiate the trainable parameters, as in Sobolev training.
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,eps=1e-7)
optimizer.zero_grad()
derivative_loss=dydx.square().mean()+dy2dx2.square().mean()
derivative_loss.backward()
assert model[0].weight.grad is not None and torch.isfinite(model[0].weight.grad).all()
assert model[0].weight.grad.norm()>0
optimizer.step()
model=save_and_check(model,optimizer,"derivatives",x_data.cpu(),factory=lambda: dense_tanh(4,5))
fig,ax=plt.subplots();im=ax.imshow(dydx[0].detach().cpu());fig.colorbar(im,ax=ax)
ax.set(xlabel="input index",ylabel="output index",title="Jacobian of first example")
fig.savefig(OUT / "jacobian.png");plt.show();plt.close(fig)
write_metrics({"jacobian_max_error":float((dydx-expected_j).abs().max().detach()),
               "hessian_max_error":float((dy2dx2-expected_h).abs().max().detach()),
               "derivative_loss":float(derivative_loss.detach())})


# Why do we need this?

- we have input $x,t$ and output $u(x,t)$ for typical spatial-temporal dynamics
- when we are using PDE information, we need things like $\partial{u}/\partial{x} + \partial{u}/\partial{t}=0$
- they can be embedded elegantly inside the `torch.nn.Module` **in the form of layer regularization loss**.